# 04 — Confidence Calibration (Platt Scaling)

**DRISHTI** — Turning raw YOLO confidence into honest calibrated probabilities

Raw YOLO confidence isn't a real probability — a model saying "85%" doesn't mean
85% of past detections at that score were actually real. Platt scaling
(LogisticRegression on raw_conf → correct/incorrect) fixes this.

**Run on:** Colab/Kaggle with GPU (for inference on val set).

In [ ]:
!pip install -q ultralytics scikit-learn matplotlib numpy opencv-python-headless

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pickle
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from ultralytics import YOLO

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
print('Setup complete.')

## 1. Collect Predictions on Validation Set

Run the trained model on every val image and record
(raw_confidence, is_correct) pairs.

In [ ]:
# ---- Point these at your trained model and data ----
MODEL_PATH = 'drishti_training/runs/yolov8s_seg_baseline/weights/best.pt'  # from notebook 02
VAL_IMAGES = Path('drishti_training/data/splits/val/images')
VAL_LABELS = Path('drishti_training/data/splits/val/labels')

model = YOLO(MODEL_PATH)
print(f'Model loaded: {MODEL_PATH}')

In [ ]:
def collect_predictions(model, images_dir, labels_dir, iou_thresh=0.5):
    """Collect (raw_confidence, is_correct) pairs."""
    confidences, correctness = [], []
    
    image_paths = sorted(images_dir.glob('*.png'))
    print(f'Running inference on {len(image_paths)} val images...')
    
    for img_path in image_paths:
        stem = img_path.stem
        lbl_path = labels_dir / f'{stem}.txt'
        
        # Load GT boxes
        gt_boxes = []
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5: continue
                    cls_id = int(parts[0])
                    coords = list(map(float, parts[1:]))
                    xs, ys = coords[0::2], coords[1::2]
                    if xs and ys:
                        gt_boxes.append({'cls': cls_id, 'x1': min(xs), 'y1': min(ys), 'x2': max(xs), 'y2': max(ys)})
        
        # Inference
        results = model.predict(str(img_path), conf=0.01, verbose=False)
        if not results or results[0].boxes is None: continue
        
        img = cv2.imread(str(img_path))
        ih, iw = img.shape[:2]
        
        for i in range(len(results[0].boxes)):
            conf = float(results[0].boxes.conf[i])
            pred_cls = int(results[0].boxes.cls[i])
            box = results[0].boxes.xyxy[i].cpu().numpy()
            pred = {'x1': box[0]/iw, 'y1': box[1]/ih, 'x2': box[2]/iw, 'y2': box[3]/ih}
            
            correct = 0
            for gt in gt_boxes:
                if gt['cls'] != pred_cls: continue
                ix1 = max(pred['x1'], gt['x1']); iy1 = max(pred['y1'], gt['y1'])
                ix2 = min(pred['x2'], gt['x2']); iy2 = min(pred['y2'], gt['y2'])
                if ix2 > ix1 and iy2 > iy1:
                    inter = (ix2-ix1)*(iy2-iy1)
                    area1 = (pred['x2']-pred['x1'])*(pred['y2']-pred['y1'])
                    area2 = (gt['x2']-gt['x1'])*(gt['y2']-gt['y1'])
                    iou = inter / (area1+area2-inter+1e-10)
                    if iou >= iou_thresh:
                        correct = 1; break
            
            confidences.append(conf)
            correctness.append(correct)
    
    return np.array(confidences), np.array(correctness)

confidences, correctness = collect_predictions(model, VAL_IMAGES, VAL_LABELS)
print(f'\nCollected {len(confidences)} predictions: {correctness.sum()} correct, {len(correctness)-correctness.sum()} incorrect')

## 2. Before Calibration — Raw Confidence Analysis

In [ ]:
def reliability_diagram(confidences, correctness, n_bins=10, title=''):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_counts = [], [], []
    
    for i in range(n_bins):
        mask = (confidences >= bin_edges[i]) & (confidences < bin_edges[i+1])
        if mask.sum() == 0:
            bin_accs.append(0); bin_confs.append(0); bin_counts.append(0)
            continue
        bin_accs.append(correctness[mask].mean())
        bin_confs.append(confidences[mask].mean())
        bin_counts.append(mask.sum())
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Reliability diagram
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    ax1.bar(centers, bin_accs, width=1/n_bins*0.8, alpha=0.6, color='steelblue', label='Accuracy')
    ax1.plot([0,1], [0,1], 'r--', label='Perfect calibration')
    ax1.set_xlabel('Confidence'); ax1.set_ylabel('Accuracy')
    ax1.set_title(f'Reliability Diagram{" — " + title if title else ""}')
    ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
    ax1.legend(); ax1.grid(True, alpha=0.3)
    
    # Count histogram
    ax2.bar(centers, bin_counts, width=1/n_bins*0.8, alpha=0.6, color='coral')
    ax2.set_xlabel('Confidence'); ax2.set_ylabel('Count')
    ax2.set_title('Prediction Distribution')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # ECE
    ece = sum(abs(a - c) * n / max(sum(bin_counts), 1) for a, c, n in zip(bin_accs, bin_confs, bin_counts))
    print(f'Expected Calibration Error (ECE): {ece:.4f}')
    return ece

print('BEFORE calibration:')
ece_before = reliability_diagram(confidences, correctness, title='Raw Confidence')

## 3. Fit Platt Scaling (LogisticRegression)

In [ ]:
# ---- Platt scaling ----
X = confidences.reshape(-1, 1)
y = correctness

calibrator = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
calibrator.fit(X, y)

# CV accuracy
cv_scores = cross_val_score(calibrator, X, y, cv=min(5, len(y)), scoring='accuracy')
print(f'Calibrator CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Coefficient: {calibrator.coef_[0][0]:.4f}')
print(f'Intercept:   {calibrator.intercept_[0]:.4f}')

# Calibrated probabilities
cal_probs = calibrator.predict_proba(X)[:, 1]

print(f'\nAFTER calibration:')
ece_after = reliability_diagram(cal_probs, correctness, title='Calibrated (Platt)')

print(f'\nECE improvement: {ece_before:.4f} → {ece_after:.4f} ({(1-ece_after/max(ece_before,1e-10))*100:.1f}% reduction)')

In [ ]:
# ---- Calibration mapping curve ----
raw_range = np.linspace(0, 1, 200).reshape(-1, 1)
cal_range = calibrator.predict_proba(raw_range)[:, 1]

plt.figure(figsize=(8, 6))
plt.plot(raw_range, cal_range, 'b-', linewidth=2, label='Platt scaling')
plt.plot([0,1], [0,1], 'r--', alpha=0.5, label='Identity (no calibration)')
plt.xlabel('Raw YOLO Confidence', fontsize=12)
plt.ylabel('Calibrated Probability', fontsize=12)
plt.title('Platt Scaling: Raw → Calibrated Confidence', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Annotate key points
for raw in [0.25, 0.5, 0.75, 0.9]:
    cal = calibrator.predict_proba([[raw]])[0][1]
    plt.annotate(f'{raw:.0%} → {cal:.0%}', xy=(raw, cal),
                 xytext=(raw-0.15, cal+0.08),
                 arrowprops=dict(arrowstyle='->', color='gray'),
                 fontsize=9, color='darkblue')

plt.tight_layout()
plt.show()

## 4. Save Calibrator

In [ ]:
output_path = Path('calibrator.pkl')
with open(output_path, 'wb') as f:
    pickle.dump(calibrator, f)

print(f'Calibrator saved to: {output_path}')
print(f'File size: {output_path.stat().st_size} bytes')
print()
print('This file is loaded by ml/inference/confidence_filter.py at serve time.')
print('Copy to ml/models/exported/calibrator.pkl for deployment.')
print()
print('Mapping preview (raw → calibrated):')
for raw in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    cal = calibrator.predict_proba([[raw]])[0][1]
    print(f'  Raw {raw:.2f} → Calibrated {cal:.2f} ({cal*100:.0f}%)')

## 5. Summary

| Metric | Before | After |
|--------|--------|-------|
| ECE | (see above) | (see above) |

The calibrated score is what turns "the model said 85%" into "85% of past
detections at this score band were actually real" — the honest version for the PS.

**Next:** `05_error_analysis.ipynb`